<a href="https://colab.research.google.com/github/shrutikhati70-hash/Datawarehouisng_and_bigdata_analytics/blob/main/Spark_streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pyspark

##Create a TCP socket server

In [5]:
import socket
import threading
import time

HOST = "127.0.0.1"
PORT = 9999

server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
server.bind((HOST, PORT))
server.listen(1)

print("Socket server started on port 9999")

def send_data():
    conn, addr = server.accept()
    print("Spark connected!")

    messages = [
        "Hello Spark",
        "Hello World",
        "Spark Streaming",
        "Big Data",
        "Hello Spark"
    ]

    for message in messages:
        conn.sendall((message + "\n").encode())
        print("Sent:", message)
        time.sleep(2)

    conn.close()

thread = threading.Thread(target=send_data, daemon=True)
thread.start()

Socket server started on port 9999


##Create the Spark Streaming program

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

# Create Spark Session
spark = SparkSession.builder \
    .appName("SparkStreamingWordCount") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Read streaming data from socket
lines = spark.readStream \
    .format("socket") \
    .option("host", "127.0.0.1") \
    .option("port", 9999) \
    .load()

# Split sentences into individual words
words = lines.select(
    explode(split(lines.value, " ")).alias("word")
)

# Apply Word Count
wordCounts = words.groupBy("word").count()

# Start streaming with checkpointing
query = wordCounts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("checkpointLocation", "/content/spark_checkpoint") \
    .start()

# Wait for 30 seconds while streaming
query.awaitTermination(30)

False

##Stop the streaming query

In [10]:
query.stop()
server.close()

print("Streaming stopped successfully!")

Streaming stopped successfully!


##Verify checkpointing

In [11]:
import os

checkpoint_dir = "/content/spark_checkpoint"

print("Checkpoint directory exists:",
      os.path.exists(checkpoint_dir))

print("Checkpoint files:",
      os.listdir(checkpoint_dir))

Checkpoint directory exists: True
Checkpoint files: ['metadata', 'offsets', '.metadata.crc', 'commits', 'state']
